In [1]:
transaction = {
    "transaction_id": "T001",
    "date": "2026-11-17",
    "product": "Notebook",
    "quantity": 10,
    "unit_price": 50.0,
    "revenue": 500.0
}


In [2]:
transactions = [
    {"transaction_id": "T001", "product": "Notebook", "quantity": 10,
     "unit_price": 50.0, "revenue": 500.0},
    {"transaction_id": "T002", "product": "Pen", "quantity": 20,
     "unit_price": 10.0, "revenue": 200.0}
]


In [3]:
START
  read input file path and revenue target
  load rows from CSV
  create empty valid_transactions list
  create empty rejected_rows list

  FOR each CSV row, keeping its row number
      validate required text fields
      validate date format
      convert quantity to integer
      convert unit price to number
      IF row is valid
          calculate revenue
          add transaction to valid_transactions
      ELSE
          add row number and reason to rejected_rows

  calculate total revenue and total units
  calculate average transaction value
  calculate revenue grouped by product
  compare total revenue with target
  write summary report
  write product revenue report
  write rejected-row report
  display completion message
END


<class 'IndentationError'>: unexpected indent (808847713.py, line 2)

In [4]:
def calculate_revenue(quantity, unit_price):
    return quantity * unit_price

amount = calculate_revenue(3, 25.0)
print(amount)  # 75.0


75.0


In [5]:
amounts = [100.0, 250.0, 50.0]
total = 0.0

for amount in amounts:
    total = total + amount

print(total)  # 400.0

400.0


In [6]:
product_revenue = {}
product = "Notebook"
revenue = 500.0

if product in product_revenue:
    product_revenue[product] += revenue
else:
    product_revenue[product] = revenue

print(product_revenue)


{'Notebook': 500.0}


In [7]:
raw_quantity = "ten"

try:
    quantity = int(raw_quantity)
except ValueError:
    print("Quantity must be a whole number.")

Quantity must be a whole number.


In [9]:
transaction_id,date,product,quantity,unit_price
T001,2026-11-17,Notebook,10,50
T002,2026-11-17,Pen,20,10
T003,2026-11-18,Notebook,5,50
T004,2026-11-18,Folder,8,25
T005,2026-11-18,Pen,12,10

<class 'NameError'>: name 'transaction_id' is not defined

In [10]:
import csv

def load_csv(file_path):
    rows = []

    with open(file_path, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            rows.append(row)

    return rows


In [11]:
if __name__ == "__main__":
    records = load_csv("data/sales_transactions.csv")
    print("Rows loaded:", len(records))
    print(records[0])


<class 'FileNotFoundError'>: [Errno 44] No such file or directory: 'data/sales_transactions.csv'

In [12]:
from datetime import datetime

def validate_transaction(row):
    required_fields = [
        "transaction_id", "date", "product", "quantity", "unit_price"
    ]

    for field in required_fields:
        value = row.get(field, "")
        if value is None or value.strip() == "":
            raise ValueError(f"Missing required field: {field}")

    try:
        datetime.strptime(row["date"], "%Y-%m-%d")
    except ValueError as error:
        raise ValueError("Date must use YYYY-MM-DD format") from error

    try:
        quantity = int(row["quantity"])
    except ValueError as error:
        raise ValueError("Quantity must be a whole number") from error

    try:
        unit_price = float(row["unit_price"])
    except ValueError as error:
        raise ValueError("Unit price must be numeric") from error

    if quantity < 0:
        raise ValueError("Quantity cannot be negative")

    if unit_price < 0:
        raise ValueError("Unit price cannot be negative")

    return {
        "transaction_id": row["transaction_id"].strip(),
        "date": row["date"].strip(),
        "product": row["product"].strip(),
        "quantity": quantity,
        "unit_price": unit_price
    }


In [13]:
def calculate_summary(transactions):
    total_revenue = 0.0
    total_units = 0
    for transaction in transactions:
        total_revenue += transaction["quantity"] * transaction["unit_price"]
        total_units += transaction["quantity"]

    transaction_count = len(transactions)
    if transaction_count == 0:
        average_transaction_value = 0.0
    else:
        average_transaction_value = total_revenue / transaction_count

    return {
        "transaction_count": transaction_count,
        "total_revenue": total_revenue,
        "total_units": total_units,
        "average_transaction_value": average_transaction_value
    }


def calculate_product_revenue(transactions):
    totals = {}
    for transaction in transactions:
        product = transaction["product"]
        revenue = transaction["quantity"] * transaction["unit_price"]

        if product not in totals:
            totals[product] = 0.0
        totals[product] += revenue

    return totals


In [14]:
import csv
from pathlib import Path

def write_summary(summary, target, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    status = "TARGET ACHIEVED" if summary["total_revenue"] >= target else "TARGET NOT ACHIEVED"

    with open(output_path, "w", encoding="utf-8") as file:
        file.write("RETAIL SALES SUMMARY\\n")
        file.write(f"Valid transactions: {summary['transaction_count']}\\n")
        file.write(f"Total units sold: {summary['total_units']}\\n")
        file.write(f"Total revenue: {summary['total_revenue']:.2f}\\n")
        file.write(f"Average transaction value: {summary['average_transaction_value']:.2f}\\n")
        file.write(f"Revenue target: {target:.2f}\\n")
        file.write(f"Target status: {status}\\n")


def write_product_report(product_totals, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["product", "revenue"])
        for product, revenue in sorted(product_totals.items()):
            writer.writerow([product, f"{revenue:.2f}"])


def write_rejected_report(rejected_rows, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["row_number", "reason"])
        for item in rejected_rows:
            writer.writerow([item["row_number"], item["reason"]])


In [15]:
from pathlib import Path
from data_loader import load_csv
from data_validator import validate_transaction
from sales_calculator import calculate_summary, calculate_product_revenue
from report_generator import (
    write_summary, write_product_report, write_rejected_report
)

def run_application():
    input_path = "data/sales_transactions.csv"
    revenue_target = 1000.0

    raw_rows = load_csv(input_path)
    valid_transactions = []
    rejected_rows = []

    # DictReader data rows start on CSV line 2 because line 1 is the header.
    for row_number, row in enumerate(raw_rows, start=2):
        try:
            transaction = validate_transaction(row)
            valid_transactions.append(transaction)
        except ValueError as error:
            rejected_rows.append({
                "row_number": row_number,
                "reason": str(error)
            })

    summary = calculate_summary(valid_transactions)
    product_totals = calculate_product_revenue(valid_transactions)

    write_summary(summary, revenue_target, "output/sales_summary.txt")
    write_product_report(product_totals, "output/product_revenue.csv")
    write_rejected_report(rejected_rows, "output/rejected_rows.csv")

    print("Processing completed.")
    print("Valid rows:", len(valid_transactions))
    print("Rejected rows:", len(rejected_rows))
    print("Total revenue:", f"{summary['total_revenue']:.2f}")
    print("Reports saved in the output folder.")

if __name__ == "__main__":
    run_application()


<class 'ModuleNotFoundError'>: No module named 'data_loader'